What happens when you DELETE or UPDATE a row in Delta Lake?

--------

Traditional Method:

Assume there are 100 rows in xyz.parquet and they are refered by delta log and delta table

If we delete 15 rows out of 100, 
Delta lake doesn't modify the original parquet file in-place. Instead:

    * It writes a new parquet file with the remaining rows 

Delta transaction log (_delta_log/) is updated to:
    * Mark the old file as removed (not physically deleted immediately)
    * Add a refrence to the new file

VACCUM removes actual xyz.parquet

-------

Latest Method : DELETION VECTORS in DBX

In DBX Delta Lake, a Deletion Vector (DV) is a mechanism to logically delete rows from a table DURING DELETE etc without physically removing them from the underlying data files/ creating new data file

Why are Deletion vectors needed?

In traditional method, rewriting new parquet files is expensive, especially with large datasets

---------

What is a Deletion Vector?

A Deletion Vector is a compact bitmap or list that keeps track of which rows in a data file are 'deleted' - without rewriting the file.

Marks the rows as 'deleted' using a deletion vector stored alongside the data file

While reading the data: Delta lake skips those rows based on the Deletion Vector

Deletion vector is stored in a small separate .bin file in the Delta log folder, and referenced in the Delta table's metadata (_delta_log JSON files)

-------

VACUUM dont delete rows from parquet files if there still rows alive

VACUUM only delete complete file, if it is completely unreferenced by delta log

--------

OPTIMIZE forms a new parquet with alive rows and old parquet becomes unreferred by delta log and gets deleted by VACUUM — This is similar to traditional method, but it doesn't happen every time rows are deleted from the table.